# 第4节：音视频容器格式与编解码

本 Notebook 包含三个实验，帮助你理解容器、码流、编解码器的概念。

**实验内容：**
1. 用 ffprobe 分析不同格式文件
2. 提取并对比裸流
3. 封装格式转换实验

## 环境准备

确保已安装 ffmpeg 和 ffprobe。

In [ ]:
import subprocess
import json
import os
import time

# 检查 ffmpeg 和 ffprobe 是否可用
def check_command(cmd):
    try:
        result = subprocess.run([cmd, '-version'], capture_output=True, text=True)
        version = result.stdout.split('\n')[0]
        print(f"✓ {cmd} 已安装: {version}")
        return True
    except FileNotFoundError:
        print(f"✗ {cmd} 未安装")
        return False

check_command('ffmpeg')
check_command('ffprobe')

## 生成测试素材

生成 5 秒的测试视频（720p，H.264+AAC）。

In [ ]:
# 生成测试视频
def run_ffmpeg_cmd(cmd, description):
    """执行 ffmpeg 命令并检查结果"""
    print(f"  {description}...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ✗ 失败: {result.stderr[:200]}")
        return False
    return True

def generate_test_files():
    """生成测试素材"""
    print("生成测试素材中...")
    
    # 生成 MP4 测试文件
    cmd_mp4 = [
        'ffmpeg',
        '-f', 'lavfi', '-i', 'testsrc=duration=5:size=1280x720:rate=30',
        '-f', 'lavfi', '-i', 'sine=frequency=440:duration=5',
        '-c:v', 'libx264', '-c:a', 'aac',
        '-shortest', '-y', 'test.mp4'
    ]
    if not run_ffmpeg_cmd(cmd_mp4, "生成 MP4"):
        return
    
    # 生成 MKV 测试文件
    cmd_mkv = ['ffmpeg', '-i', 'test.mp4', '-c', 'copy', '-y', 'test.mkv']
    if not run_ffmpeg_cmd(cmd_mkv, "生成 MKV"):
        return
    
    # 提取 H.264 裸流
    cmd_h264 = ['ffmpeg', '-i', 'test.mp4', '-c:v', 'copy', '-an', '-y', 'test.h264']
    if not run_ffmpeg_cmd(cmd_h264, "提取 H.264 裸流"):
        return
    
    # 提取 AAC 裸流
    cmd_aac = ['ffmpeg', '-i', 'test.mp4', '-c:a', 'copy', '-vn', '-y', 'test.aac']
    if not run_ffmpeg_cmd(cmd_aac, "提取 AAC 裸流"):
        return
    
    # 验证文件生成
    print("\n生成结果:")
    files = ['test.mp4', 'test.mkv', 'test.h264', 'test.aac']
    for f in files:
        if os.path.exists(f):
            size = os.path.getsize(f) / 1024
            print(f"✓ {f}: {size:.1f} KB")
        else:
            print(f"✗ {f}: 生成失败")

generate_test_files()

## 实验1：用 ffprobe 分析不同格式文件

**目标**：理解 ffprobe 输出结构，识别容器、编码、流信息

In [ ]:
def probe_file(file_path):
    """使用 ffprobe 分析媒体文件"""
    if not os.path.exists(file_path):
        print(f"✗ 文件不存在: {file_path}")
        return None
    
    cmd = [
        'ffprobe',
        '-v', 'quiet',
        '-print_format', 'json',
        '-show_format',
        '-show_streams',
        file_path
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"✗ ffprobe 失败: {result.stderr[:200]}")
        return None
    
    return json.loads(result.stdout)

def print_stream_info(stream):
    """打印流信息"""
    codec_type = stream['codec_type']
    codec_name = stream['codec_name']
    
    if codec_type == 'video':
        print(f"  视频流:")
        print(f"    编码: {codec_name}")
        print(f"    分辨率: {stream['width']}x{stream['height']}")
        print(f"    帧率: {stream['r_frame_rate']}")
        print(f"    像素格式: {stream.get('pix_fmt', 'N/A')}")
    elif codec_type == 'audio':
        print(f"  音频流:")
        print(f"    编码: {codec_name}")
        print(f"    采样率: {stream['sample_rate']} Hz")
        print(f"    声道数: {stream['channels']}")
        print(f"    采样格式: {stream.get('sample_fmt', 'N/A')}")

# 分析 MP4 文件
print("=" * 50)
print("MP4 文件分析")
print("=" * 50)

mp4_info = probe_file('test.mp4')
if mp4_info:
    print(f"\n容器格式: {mp4_info['format']['format_name']}")
    print(f"时长: {float(mp4_info['format']['duration']):.2f} 秒")
    print(f"总码率: {int(mp4_info['format']['bit_rate']) / 1000:.0f} kbps")
    print(f"流数量: {mp4_info['format']['nb_streams']}")
    
    for stream in mp4_info['streams']:
        print_stream_info(stream)

# 分析 MKV 文件
print("\n" + "=" * 50)
print("MKV 文件分析")
print("=" * 50)

mkv_info = probe_file('test.mkv')
if mkv_info:
    print(f"\n容器格式: {mkv_info['format']['format_name']}")
    print(f"时长: {float(mkv_info['format']['duration']):.2f} 秒")
    print(f"总码率: {int(mkv_info['format']['bit_rate']) / 1000:.0f} kbps")
    print(f"流数量: {mkv_info['format']['nb_streams']}")
    
    for stream in mkv_info['streams']:
        print_stream_info(stream)

## 实验2：提取并对比裸流

**目标**：理解容器封装与裸流的区别

In [ ]:
def extract_streams(input_file, output_prefix):
    """从容器中提取裸流"""
    if not os.path.exists(input_file):
        print(f"✗ 输入文件不存在: {input_file}")
        return None, None, None
    
    # 提取视频流
    cmd_video = [
        'ffmpeg', '-i', input_file,
        '-c:v', 'copy', '-an',
        '-y', f'{output_prefix}_video.h264'
    ]
    
    # 提取音频流
    cmd_audio = [
        'ffmpeg', '-i', input_file,
        '-c:a', 'copy', '-vn',
        '-y', f'{output_prefix}_audio.aac'
    ]
    
    result_v = subprocess.run(cmd_video, capture_output=True, text=True)
    if result_v.returncode != 0:
        print(f"✗ 提取视频流失败: {result_v.stderr[:200]}")
        return None, None, None
    
    result_a = subprocess.run(cmd_audio, capture_output=True, text=True)
    if result_a.returncode != 0:
        print(f"✗ 提取音频流失败: {result_a.stderr[:200]}")
        return None, None, None
    
    # 获取文件大小
    original_size = os.path.getsize(input_file)
    video_size = os.path.getsize(f'{output_prefix}_video.h264')
    audio_size = os.path.getsize(f'{output_prefix}_audio.aac')
    
    return original_size, video_size, audio_size

# 从 MP4 提取裸流
print("=" * 50)
print("从 MP4 提取裸流")
print("=" * 50)

orig_size, video_size, audio_size = extract_streams('test.mp4', 'extracted')

if orig_size is not None:
    print(f"\n原始 MP4 文件: {orig_size / 1024:.1f} KB")
    print(f"H.264 裸流:    {video_size / 1024:.1f} KB")
    print(f"AAC 裸流:      {audio_size / 1024:.1f} KB")
    print(f"裸流总和:      {(video_size + audio_size) / 1024:.1f} KB")
    print(f"容器开销:      {(orig_size - video_size - audio_size) / 1024:.1f} KB")

# 分析裸流
print("\n" + "=" * 50)
print("裸流文件信息")
print("=" * 50)

for filename in ['extracted_video.h264', 'extracted_audio.aac']:
    if os.path.exists(filename):
        size = os.path.getsize(filename) / 1024
        print(f"\n{filename}: {size:.1f} KB")
        
        # 使用 ffprobe 分析裸流
        info = probe_file(filename)
        if info and 'streams' in info and len(info['streams']) > 0:
            stream = info['streams'][0]
            print(f"  编码: {stream.get('codec_name', 'N/A')}")
            if stream['codec_type'] == 'video':
                print(f"  分辨率: {stream.get('width', 'N/A')}x{stream.get('height', 'N/A')}")
            elif stream['codec_type'] == 'audio':
                print(f"  采样率: {stream.get('sample_rate', 'N/A')} Hz")

## 实验3：封装格式转换实验

**目标**：理解转封装与转码的区别

In [ ]:
def remux_file(input_file, output_file):
    """转封装：只更换容器，不改变码流"""
    if not os.path.exists(input_file):
        print(f"✗ 输入文件不存在: {input_file}")
        return None, None
    
    cmd = [
        'ffmpeg', '-i', input_file,
        '-c', 'copy',  # 关键：copy 表示不重新编码
        '-y', output_file
    ]
    
    start = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - start
    
    if result.returncode != 0:
        print(f"✗ 转封装失败: {result.stderr[:200]}")
        return None, None
    
    return elapsed, result.returncode

def transcode_file(input_file, output_file):
    """转码：重新编码码流"""
    if not os.path.exists(input_file):
        print(f"✗ 输入文件不存在: {input_file}")
        return None, None
    
    cmd = [
        'ffmpeg', '-i', input_file,
        '-c:v', 'libx264',  # 重新编码视频
        '-c:a', 'aac',      # 重新编码音频
        '-y', output_file
    ]
    
    start = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - start
    
    if result.returncode != 0:
        print(f"✗ 转码失败: {result.stderr[:200]}")
        return None, None
    
    return elapsed, result.returncode

# 对比转封装和转码
print("=" * 50)
print("转封装 vs 转码对比")
print("=" * 50)

# 转封装
remux_time, remux_ret = remux_file('test.mkv', 'remuxed.mp4')
if remux_time is not None:
    remux_size = os.path.getsize('remuxed.mp4') / 1024
    print(f"\n转封装 (MKV → MP4):")
    print(f"  耗时: {remux_time:.3f} 秒")
    print(f"  输出大小: {remux_size:.1f} KB")
    print(f"  返回码: {remux_ret}")

# 转码
transcode_time, transcode_ret = transcode_file('test.mkv', 'transcoded.mp4')
if transcode_time is not None:
    transcode_size = os.path.getsize('transcoded.mp4') / 1024
    print(f"\n转码 (MKV → MP4):")
    print(f"  耗时: {transcode_time:.3f} 秒")
    print(f"  输出大小: {transcode_size:.1f} KB")
    print(f"  返回码: {transcode_ret}")

# 对比分析
if remux_time is not None and transcode_time is not None:
    print("\n" + "=" * 50)
    print("对比分析")
    print("=" * 50)
    
    # 防止除零错误
    if remux_time > 0:
        speed_ratio = transcode_time / remux_time
        print(f"\n速度比: 转封装比转码快 {speed_ratio:.1f} 倍")
    else:
        print(f"\n速度比: 转封装耗时过短，无法计算")
    
    print(f"大小比: 转封装 {remux_size:.1f} KB vs 转码 {transcode_size:.1f} KB")
    
    # 验证转封装的码流是否相同
    print("\n验证码流是否相同:")
    orig_info = probe_file('test.mkv')
    remux_info = probe_file('remuxed.mp4')
    
    if orig_info and remux_info:
        for i, (orig_stream, remux_stream) in enumerate(zip(orig_info['streams'], remux_info['streams'])):
            if orig_stream['codec_name'] == remux_stream['codec_name']:
                print(f"  流 {i}: 编码相同 ({orig_stream['codec_name']})")
            else:
                print(f"  流 {i}: 编码不同 ({orig_stream['codec_name']} vs {remux_stream['codec_name']})")

## 总结

通过本实验，你应该理解了：

1. **容器、码流、编解码器的区别**
   - 容器是包装盒（MP4/MKV/FLV）
   - 码流是内容（H.264/AAC）
   - 编解码器是算法（x264/fdk-aac）

2. **裸流与容器文件的差异**
   - 裸流只包含码流数据，没有元信息
   - 容器文件包含码流 + 元信息 + 索引

3. **转封装与转码的区别**
   - 转封装：只换容器，速度快，码流不变
   - 转码：重新编码，速度慢，可改变参数

4. **ffprobe 的使用**
   - 分析容器格式、流信息、编码参数
   - JSON 输出便于程序解析